# Modelo de Clasificación para obtener si deacuerdo a sus caracteristicas el alumno solicitara un tutor

In [251]:
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import r2_score, mean_absolute_error, roc_auc_score, accuracy_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, chi2, f_regression, mutual_info_regression
from sklearn.linear_model import LinearRegression, LogisticRegression, Lars, Ridge, Lasso, ElasticNet, BayesianRidge, SGDRegressor, SGDClassifier
from sklearn.impute import SimpleImputer
from scipy.stats import ks_2samp
from sklearn.feature_selection import VarianceThreshold
from varclushi import VarClusHi
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_graphviz
pd.set_option('display.float_format', lambda x: "{:,.2f}".format(x))

In [3]:
ruta= os.path.join('/Users/whiz/Downloads/', 'enhanced_student_habits_performance_dataset.csv')

In [4]:
df = pd.read_csv(ruta, sep=',')

In [5]:
df.shape

(80000, 31)

In [6]:
df.head(5)

,student_id,age,gender,major,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,...,screen_time,study_environment,access_to_tutoring,family_income_range,parental_support_level,motivation_level,exam_anxiety_score,learning_style,time_management_score,exam_score
0,100000,26,Male,Computer Science,7.65,3.00,0.10,Yes,70.30,6.20,...,10.90,Co-Learning Group,Yes,High,9,7,8,Reading,3.00,100
1,100001,28,Male,Arts,5.70,0.50,0.40,No,88.40,7.20,...,8.30,Co-Learning Group,Yes,Low,7,2,10,Reading,6.00,99
2,100002,17,Male,Arts,2.40,4.20,0.70,No,82.10,9.20,...,8.00,Library,Yes,High,3,9,6,Kinesthetic,7.60,98
3,100003,27,Other,Psychology,3.40,4.60,2.30,Yes,79.30,4.20,...,11.70,Co-Learning Group,Yes,Low,5,3,10,Reading,3.20,100
4,100004,25,Female,Business,4.70,0.80,2.70,Yes,62.90,6.50,...,9.40,Quiet Room,Yes,Medium,9,1,10,Reading,7.10,98


In [7]:
df.describe([0.01, 0.99])

,student_id,age,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,exercise_frequency,mental_health_rating,previous_gpa,semester,stress_level,social_activity,screen_time,parental_support_level,motivation_level,exam_anxiety_score,time_management_score,exam_score
count,"80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00"
mean,"139,999.50",22.00,4.17,2.50,2.00,69.97,7.02,3.52,6.80,3.60,4.50,5.01,2.50,9.67,5.48,5.49,8.51,5.50,89.14
std,"23,094.16",3.75,2.00,1.45,1.16,17.33,1.47,2.29,1.92,0.46,2.30,1.95,1.70,2.78,2.87,2.87,1.80,2.60,11.59
min,"100,000.00",16.00,0.00,0.00,0.00,40.00,4.00,0.00,1.00,1.64,1.00,1.00,0.00,0.30,1.00,1.00,5.00,1.00,36.00
1%,"100,799.99",16.00,0.00,0.10,0.00,40.60,4.00,0.00,2.10,2.37,1.00,1.00,0.00,3.40,1.00,1.00,5.00,1.10,57.00
50%,"139,999.50",22.00,4.13,2.50,2.00,69.90,7.00,4.00,6.90,3.79,5.00,5.00,2.00,9.70,5.00,5.00,10.00,5.50,93.00
99%,"179,199.01",28.00,8.90,5.00,4.00,99.40,10.50,7.00,10.00,4.00,8.00,9.70,5.00,16.10,10.00,10.00,10.00,9.90,100.00
max,"179,999.00",28.00,12.00,5.00,4.00,100.00,12.00,7.00,10.00,4.00,8.00,10.00,5.00,21.00,10.00,10.00,10.00,10.00,100.00


In [8]:
df.dtypes

student_id                         int64
age                                int64
gender                            object
major                             object
study_hours_per_day              float64
social_media_hours               float64
netflix_hours                    float64
part_time_job                     object
attendance_percentage            float64
sleep_hours                      float64
diet_quality                      object
exercise_frequency                 int64
parental_education_level          object
internet_quality                  object
mental_health_rating             float64
extracurricular_participation     object
previous_gpa                     float64
semester                           int64
stress_level                     float64
dropout_risk                      object
social_activity                    int64
screen_time                      float64
study_environment                 object
access_to_tutoring                object
family_income_ra

In [9]:
df['tutor'] = df['access_to_tutoring'].map({"Yes": 1, "No": 0})

In [10]:
df[['tutor']].value_counts()

tutor
0        40039
1        39961
Name: count, dtype: int64

In [11]:
df.drop('access_to_tutoring', axis=1, inplace=True)

### Variables

In [13]:
um = ['student_id']
tgt = ['tutor']

In [14]:
cols_num = []
cols_str = []

In [15]:
cols_num = df.select_dtypes( include = ['float'] ).columns.tolist()
cols_num = cols_num + df.select_dtypes( include = ['int'] ).columns.tolist()

In [16]:
cols_str = df.select_dtypes( include = ['object'] ).columns.tolist()

In [17]:
df.shape

(80000, 31)

In [18]:
cols_num = [c for c in cols_num if c not in tgt+um]
cols_str = [c for c in cols_str if c not in tgt+um]

In [19]:
len(cols_num) + len(cols_str) + 2 # ese dos es um y tgt

31

In [20]:
df_final = df[ um + cols_num + cols_str + tgt ]

In [21]:
df_final.sample(1)

,student_id,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,mental_health_rating,previous_gpa,stress_level,screen_time,...,part_time_job,diet_quality,parental_education_level,internet_quality,extracurricular_participation,dropout_risk,study_environment,family_income_range,learning_style,tutor
17534,117534,3.80,3.90,0.30,92.10,8.70,7.20,2.61,3.40,9.20,...,Yes,Good,Bachelor,Low,No,No,Dorm,High,Auditory,0


# Ingeniería de Variables

In [23]:
df_final['horas_de_ocio'] = df_final['social_media_hours'] + df_final['netflix_hours']
##df_final['ratio_estudio_ocio'] = df_final['study_hours_per_day'] / (df_final['horas_de_ocio']).replace([np.inf, -np.inf, np.nan], 0)
df_final['actividad_total_diaria'] = df_final['study_hours_per_day'] + df_final['horas_de_ocio'] + df_final['sleep_hours']
df_final['manejo_de'] = df_final['time_management_score'] * df_final['study_hours_per_day']
df_final['ratio_productividad_en_pantalla'] = (df_final['screen_time'] - df_final['social_media_hours'] - df_final['netflix_hours']) / (df_final['screen_time'] + 1)
df_final['estres_ansiedad'] = df_final['stress_level'] + df_final['exam_anxiety_score']
df_final['manejo_de_estres'] = df_final['exam_anxiety_score'] - df_final['motivation_level']
df_final['ratio_acceso_apoyo'] = df_final['parental_support_level'] / (df_final['tutor'] + 1)
df_final['aprovecamiento'] = df_final['previous_gpa'] * df_final['attendance_percentage']

/var/folders/4c/db6g7k8s695bk1p_f9sd6kfc0000gn/T/ipykernel_1299/1496338845.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['horas_de_ocio'] = df_final['social_media_hours'] + df_final['netflix_hours']
/var/folders/4c/db6g7k8s695bk1p_f9sd6kfc0000gn/T/ipykernel_1299/1496338845.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['actividad_total_diaria'] = df_final['study_hours_per_day'] + df_final['horas_de_ocio'] + df_final['sleep_hours']
/var/folders/4c/db6g7k8s695bk1p_f9sd6kfc0000gn/T

In [24]:
df_final['ratio_productividad_en_pantalla'] = (df_final['screen_time'] - df_final['social_media_hours'] - df_final['netflix_hours']) / (df_final['screen_time'] + 1)
df_final['estres_ansiedad'] = df_final['stress_level'] + df_final['exam_anxiety_score']
df_final['manejo_de_estres'] = df_final['exam_anxiety_score'] - df_final['motivation_level']
df_final['aprovecamiento'] = df_final['previous_gpa'] * df_final['attendance_percentage']
diet_quality_mapping = {'Poor': 1, 'Fair': 2, 'Good': 3, 'Excellent': 4}
df_final['diet_quality_encoded'] = df_final['diet_quality'].map(diet_quality_mapping)
part_time_job_mapping = {'No': 0, 'Yes': 1}
df_final['part_time_job_encoded'] = df_final['part_time_job'].map(part_time_job_mapping)
extracurricular_participation_mapping = {'No': 0, 'Yes': 1}
df_final['extracurricular_participation_encoded'] = df_final['extracurricular_participation'].map(extracurricular_participation_mapping)
internet_quality_mapping = {'Low': 1, 'Medium': 2, 'High': 3}
df_final['internet_quality_encoded'] = df_final['internet_quality'].map(internet_quality_mapping)
study_environment_mapping = {
    'Quiet Room': 3,
    'Library': 2,
    'Dorm': 1,
    'Co-Learning Group': 2, 
    'Cafe': 1 
}
df_final['horas_productivas_netas'] = df_final['study_hours_per_day'] - df_final['horas_de_ocio']
df_final['tiempo_pantalla_vs_productividad'] = df_final['screen_time'] / (df_final['study_hours_per_day'].replace(0, np.nan) + 0.000001)
df_final['tiempo_pantalla_vs_productividad'] = df_final['tiempo_pantalla_vs_productividad'].fillna(0) # Rellenar NaN con 0

/var/folders/4c/db6g7k8s695bk1p_f9sd6kfc0000gn/T/ipykernel_1299/2260320881.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['ratio_productividad_en_pantalla'] = (df_final['screen_time'] - df_final['social_media_hours'] - df_final['netflix_hours']) / (df_final['screen_time'] + 1)
/var/folders/4c/db6g7k8s695bk1p_f9sd6kfc0000gn/T/ipykernel_1299/2260320881.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['estres_ansiedad'] = df_final['stress_level'] + df_final['exam_anxiety_score']
/var/

In [25]:
df_final[['tutor']].value_counts()

tutor
0        40039
1        39961
Name: count, dtype: int64

In [26]:
df_final.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['student_id', 'study_hours_per_day', 'social_media_hours',
       'netflix_hours', 'attendance_percentage', 'sleep_hours',
       'mental_health_rating', 'previous_gpa', 'stress_level', 'screen_time',
       'time_management_score', 'age', 'exercise_frequency', 'semester',
       'social_activity', 'parental_support_level', 'motivation_level',
       'exam_anxiety_score', 'exam_score', 'gender', 'major', 'part_time_job',
       'diet_quality', 'parental_education_level', 'internet_quality',
       'extracurricular_participation', 'dropout_risk', 'study_environment',
       'family_income_range', 'learning_style', 'tutor', 'horas_de_ocio',
       'actividad_total_diaria', 'manejo_de',
       'ratio_productividad_en_pantalla', 'estres_ansiedad',
       'manejo_de_estres', 'ratio_acceso_apoyo', 'aprovecamiento',
       'diet_quality_encoded', 'part_time_job_encoded',
       'extracurricular_participation_encoded', 'internet_quality_encoded',
  

In [27]:
cols_num = cols_num + ['horas_de_ocio',
       'actividad_total_diaria', 'manejo_de',
       'ratio_productividad_en_pantalla', 'estres_ansiedad',
       'manejo_de_estres', 'ratio_acceso_apoyo', 'aprovecamiento',
       'diet_quality_encoded', 'part_time_job_encoded',
       'extracurricular_participation_encoded', 'internet_quality_encoded',
       'horas_productivas_netas', 'tiempo_pantalla_vs_productividad']                

In [28]:
df_final.isna().sum()

student_id                               0
study_hours_per_day                      0
social_media_hours                       0
netflix_hours                            0
attendance_percentage                    0
sleep_hours                              0
mental_health_rating                     0
previous_gpa                             0
stress_level                             0
screen_time                              0
time_management_score                    0
age                                      0
exercise_frequency                       0
semester                                 0
social_activity                          0
parental_support_level                   0
motivation_level                         0
exam_anxiety_score                       0
exam_score                               0
gender                                   0
major                                    0
part_time_job                            0
diet_quality                             0
parental_ed

In [29]:
df_final.shape

(80000, 45)

In [30]:
len(cols_num),len(cols_str)

(32, 11)

# Análisis Exploratorio

## Categóricas

In [33]:
### La falta de información también es información para las categóricas
for c in cols_str:
    df_final[c] = df_final[c].fillna("Sin categoría")

/var/folders/4c/db6g7k8s695bk1p_f9sd6kfc0000gn/T/ipykernel_1299/2542936986.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final[c] = df_final[c].fillna("Sin categoría")


## Análisis de Frecuencias

In [35]:
def freq(df, var):
    if type(var) != list:
        var = [var]
    for v in var:
        aux = df[v].value_counts().to_frame().rename(columns={'count':'FA'})
        aux['FR'] = aux['FA'] / aux['FA'].sum()
        aux[['FAA','FRA']] = aux.apply( np.cumsum )
        print(f'Tabla de frecuencias para la variable {v} \n')
        print(aux,'\n') 

In [36]:
for c in cols_str:
    freq( df_final, c )

Tabla de frecuencias para la variable gender 

           FA   FR    FAA  FRA
gender                        
Female  26705 0.33  26705 0.33
Male    26698 0.33  53403 0.67
Other   26597 0.33  80000 1.00 

Tabla de frecuencias para la variable major 

                     FA   FR    FAA  FRA
major                                   
Arts              13505 0.17  13505 0.17
Psychology        13437 0.17  26942 0.34
Computer Science  13352 0.17  40294 0.50
Business          13276 0.17  53570 0.67
Engineering       13229 0.17  66799 0.83
Biology           13201 0.17  80000 1.00 

Tabla de frecuencias para la variable part_time_job 

                  FA   FR    FAA  FRA
part_time_job                        
No             40195 0.50  40195 0.50
Yes            39805 0.50  80000 1.00 

Tabla de frecuencias para la variable diet_quality 

                 FA   FR    FAA  FRA
diet_quality                        
Good          39935 0.50  39935 0.50
Fair          26713 0.33  66648 0.83
Poor       

## Normalización

In [38]:
def norm(df, v, umbral):
    aux = df[v].value_counts(True).to_frame()    
    aux[f'n_{v}'] = np.where( aux['proportion'] < umbral , 'CAT_PEQUEÑAS', aux.index )
    valor = aux.head(1)[f'n_{v}'].values[0]
    if aux.loc[aux[f'n_{v}'] == 'CAT_PEQUEÑAS']['proportion'].sum() < umbral:
        aux[f'n_{v}'].replace( {'CAT_PEQUEÑAS':valor} , inplace=True )
    aux.drop('proportion',axis=1, inplace=True)
    aux.reset_index(inplace=True)
               
    return df.merge(aux, left_on=[v], right_on = [v], how='left').drop(v,axis=1)  

In [39]:
for c in cols_str:
    df_final = norm( df_final, c,  0.03 )

In [40]:
cols_norm = [c for c in df_final.columns.tolist() if c[:2] == 'n_']

In [41]:
for c in cols_norm:
    freq( df_final, c )

Tabla de frecuencias para la variable n_gender 

             FA   FR    FAA  FRA
n_gender                        
Female    26705 0.33  26705 0.33
Male      26698 0.33  53403 0.67
Other     26597 0.33  80000 1.00 

Tabla de frecuencias para la variable n_major 

                     FA   FR    FAA  FRA
n_major                                 
Arts              13505 0.17  13505 0.17
Psychology        13437 0.17  26942 0.34
Computer Science  13352 0.17  40294 0.50
Business          13276 0.17  53570 0.67
Engineering       13229 0.17  66799 0.83
Biology           13201 0.17  80000 1.00 

Tabla de frecuencias para la variable n_part_time_job 

                    FA   FR    FAA  FRA
n_part_time_job                        
No               40195 0.50  40195 0.50
Yes              39805 0.50  80000 1.00 

Tabla de frecuencias para la variable n_diet_quality 

                   FA   FR    FAA  FRA
n_diet_quality                        
Good            39935 0.50  39935 0.50
Fair            

In [42]:
len(cols_norm),cols_norm

(11,
 ['n_gender',
  'n_major',
  'n_part_time_job',
  'n_diet_quality',
  'n_parental_education_level',
  'n_internet_quality',
  'n_extracurricular_participation',
  'n_dropout_risk',
  'n_study_environment',
  'n_family_income_range',
  'n_learning_style'])

In [43]:
df_final.shape

(80000, 45)

## Unarias

In [45]:
unarias = [ v for v, conteo in zip( cols_norm  , [df_final[v2].unique().shape[0] for v2 in cols_norm ] ) if conteo == 1 ]

In [46]:
len(unarias), unarias

(1, ['n_dropout_risk'])

In [47]:
len(cols_norm)

11

In [48]:
cols_norm = [c for c in cols_norm if c not in unarias]

In [49]:
len(cols_norm)

10

## Numéricas

In [51]:
X = df_final[cols_num]

In [52]:
X.shape

(80000, 32)

In [53]:
X.head()

,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,mental_health_rating,previous_gpa,stress_level,screen_time,time_management_score,...,estres_ansiedad,manejo_de_estres,ratio_acceso_apoyo,aprovecamiento,diet_quality_encoded,part_time_job_encoded,extracurricular_participation_encoded,internet_quality_encoded,horas_productivas_netas,tiempo_pantalla_vs_productividad
0,7.65,3.00,0.10,70.30,6.20,6.00,4.00,5.80,10.90,3.00,...,13.80,1,4.50,281.20,1,1,1,3,4.55,1.43
1,5.70,0.50,0.40,88.40,7.20,6.80,4.00,5.80,8.30,6.00,...,15.80,8,3.50,353.60,3,0,0,1,4.80,1.46
2,2.40,4.20,0.70,82.10,9.20,5.70,3.79,8.00,8.00,7.60,...,14.00,-3,1.50,311.16,3,0,1,1,-2.50,3.33
3,3.40,4.60,2.30,79.30,4.20,8.50,4.00,4.60,11.70,3.20,...,14.60,7,2.50,317.20,2,1,1,2,-3.50,3.44
4,4.70,0.80,2.70,62.90,6.50,9.20,4.00,5.70,9.40,7.10,...,15.70,9,4.50,251.60,3,1,0,1,1.20,2.00


## Análisis Univariado

In [55]:
X.describe([0.01, 0.99])

,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,mental_health_rating,previous_gpa,stress_level,screen_time,time_management_score,...,estres_ansiedad,manejo_de_estres,ratio_acceso_apoyo,aprovecamiento,diet_quality_encoded,part_time_job_encoded,extracurricular_participation_encoded,internet_quality_encoded,horas_productivas_netas,tiempo_pantalla_vs_productividad
count,"80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00",...,"80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00","80,000.00"
mean,4.17,2.50,2.00,69.97,7.02,6.80,3.60,5.01,9.67,5.50,...,13.52,3.02,4.12,252.08,2.33,0.50,0.50,2.00,-0.32,3.03
std,2.00,1.45,1.16,17.33,1.47,1.92,0.46,1.95,2.78,2.60,...,2.65,4.57,2.67,70.83,0.75,0.50,0.50,0.82,2.73,9.59
min,0.00,0.00,0.00,40.00,4.00,1.00,1.64,1.00,0.30,1.00,...,6.00,-5.00,0.50,78.06,1.00,0.00,0.00,1.00,-9.00,0.00
1%,0.00,0.10,0.00,40.60,4.00,2.10,2.37,1.00,3.40,1.10,...,7.10,-5.00,0.50,119.92,1.00,0.00,0.00,1.00,-6.40,0.00
50%,4.13,2.50,2.00,69.90,7.00,6.90,3.79,5.00,9.70,5.50,...,13.70,5.00,3.50,248.00,2.00,0.00,0.00,2.00,-0.37,2.28
99%,8.90,5.00,4.00,99.40,10.50,10.00,4.00,9.70,16.10,9.90,...,19.20,9.00,10.00,394.40,3.00,1.00,1.00,3.00,6.00,16.00
max,12.00,5.00,4.00,100.00,12.00,10.00,4.00,10.00,21.00,10.00,...,20.00,9.00,10.00,400.00,3.00,1.00,1.00,3.00,10.30,"2,362.00"


In [56]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_data = encoder.fit_transform(df_final[cols_norm] )

In [57]:
encoded_data

array([[0., 1., 0., ..., 0., 1., 0.],
       [0., 1., 0., ..., 0., 1., 0.],
       [0., 1., 0., ..., 1., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 1.],
       [0., 0., 1., ..., 1., 0., 0.],
       [0., 0., 1., ..., 0., 1., 0.]])

In [58]:
new_names = encoder.get_feature_names_out(cols_norm)

In [59]:
# Crea el DataFrame con los nombres correctos
df_encoded = pd.DataFrame(encoded_data, columns=new_names)
print(df_encoded)

       n_gender_Female  n_gender_Male  n_gender_Other  n_major_Arts  \
0                 0.00           1.00            0.00          0.00   
1                 0.00           1.00            0.00          1.00   
2                 0.00           1.00            0.00          1.00   
3                 0.00           0.00            1.00          0.00   
4                 1.00           0.00            0.00          0.00   
...                ...            ...             ...           ...   
79995             0.00           1.00            0.00          0.00   
79996             1.00           0.00            0.00          0.00   
79997             1.00           0.00            0.00          1.00   
79998             0.00           0.00            1.00          0.00   
79999             0.00           0.00            1.00          0.00   

       n_major_Biology  n_major_Business  n_major_Computer Science  \
0                 0.00              0.00                      1.00   
1      

In [60]:
df_f = pd.concat( [df_final,df_encoded]  ,axis=1)

In [61]:
df_f 

,student_id,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,mental_health_rating,previous_gpa,stress_level,screen_time,...,n_study_environment_Dorm,n_study_environment_Library,n_study_environment_Quiet Room,n_family_income_range_High,n_family_income_range_Low,n_family_income_range_Medium,n_learning_style_Auditory,n_learning_style_Kinesthetic,n_learning_style_Reading,n_learning_style_Visual
0,100000,7.65,3.00,0.10,70.30,6.20,6.00,4.00,5.80,10.90,...,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00
1,100001,5.70,0.50,0.40,88.40,7.20,6.80,4.00,5.80,8.30,...,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00
2,100002,2.40,4.20,0.70,82.10,9.20,5.70,3.79,8.00,8.00,...,0.00,1.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00
3,100003,3.40,4.60,2.30,79.30,4.20,8.50,4.00,4.60,11.70,...,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00
4,100004,4.70,0.80,2.70,62.90,6.50,9.20,4.00,5.70,9.40,...,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79995,179995,3.70,2.10,1.00,80.80,6.10,1.00,3.40,2.10,8.30,...,0.00,1.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00
79996,179996,1.20,0.40,2.90,99.50,4.10,5.70,2.26,3.90,4.70,...,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00
79997,179997,4.10,1.60,1.60,46.10,8.30,6.70,3.15,5.60,7.50,...,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00
79998,179998,3.80,0.60,3.50,58.70,5.80,7.60,3.67,2.40,9.30,...,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00


In [62]:
df_f.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 81 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   student_id                               80000 non-null  int64  
 1   study_hours_per_day                      80000 non-null  float64
 2   social_media_hours                       80000 non-null  float64
 3   netflix_hours                            80000 non-null  float64
 4   attendance_percentage                    80000 non-null  float64
 5   sleep_hours                              80000 non-null  float64
 6   mental_health_rating                     80000 non-null  float64
 7   previous_gpa                             80000 non-null  float64
 8   stress_level                             80000 non-null  float64
 9   screen_time                              80000 non-null  float64
 10  time_management_score                    80000

In [63]:
cols_norm

['n_gender',
 'n_major',
 'n_part_time_job',
 'n_diet_quality',
 'n_parental_education_level',
 'n_internet_quality',
 'n_extracurricular_participation',
 'n_study_environment',
 'n_family_income_range',
 'n_learning_style']

In [64]:
df_f = df_f.drop(columns=['n_gender',
 'n_major',
 'n_part_time_job',
 'n_diet_quality',
 'n_parental_education_level',
 'n_internet_quality',
 'n_extracurricular_participation',
 'n_study_environment',
 'n_family_income_range',
 'n_learning_style','n_dropout_risk' ])

In [65]:
df_f.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['student_id', 'study_hours_per_day', 'social_media_hours',
       'netflix_hours', 'attendance_percentage', 'sleep_hours',
       'mental_health_rating', 'previous_gpa', 'stress_level', 'screen_time',
       'time_management_score', 'age', 'exercise_frequency', 'semester',
       'social_activity', 'parental_support_level', 'motivation_level',
       'exam_anxiety_score', 'exam_score', 'tutor', 'horas_de_ocio',
       'actividad_total_diaria', 'manejo_de',
       'ratio_productividad_en_pantalla', 'estres_ansiedad',
       'manejo_de_estres', 'ratio_acceso_apoyo', 'aprovecamiento',
       'diet_quality_encoded', 'part_time_job_encoded',
       'extracurricular_participation_encoded', 'internet_quality_encoded',
       'horas_productivas_netas', 'tiempo_pantalla_vs_productividad',
       'n_gender_Female', 'n_gender_Male', 'n_gender_Other', 'n_major_Arts',
       'n_major_Biology', 'n_major_Business', 'n_major_Computer Science',
       'n_maj

In [66]:
X = df_f[['study_hours_per_day', 'social_media_hours',
       'netflix_hours', 'attendance_percentage', 'sleep_hours',
       'mental_health_rating', 'previous_gpa', 'stress_level', 'screen_time',
       'time_management_score', 'age', 'exercise_frequency', 'semester',
       'social_activity', 'parental_support_level', 'motivation_level',
       'exam_anxiety_score', 'exam_score','horas_de_ocio',
       'actividad_total_diaria', 'manejo_de',
       'ratio_productividad_en_pantalla', 'estres_ansiedad',
       'manejo_de_estres', 'ratio_acceso_apoyo', 'aprovecamiento',
       'n_gender_Female', 'n_gender_Male', 'n_gender_Other', 'n_major_Arts',
       'n_major_Biology', 'n_major_Business', 'n_major_Computer Science',
       'n_major_Engineering', 'n_major_Psychology', 'n_part_time_job_No',
       'n_part_time_job_Yes', 'n_diet_quality_Fair', 'n_diet_quality_Good',
       'n_diet_quality_Poor', 'n_parental_education_level_Bachelor',
       'n_parental_education_level_High School',
       'n_parental_education_level_Master', 'n_parental_education_level_PhD',
       'n_parental_education_level_Some College', 'n_internet_quality_High',
       'n_internet_quality_Low', 'n_internet_quality_Medium',
       'n_extracurricular_participation_No',
       'n_extracurricular_participation_Yes', 'n_study_environment_Cafe',
       'n_study_environment_Co-Learning Group', 'n_study_environment_Dorm',
       'n_study_environment_Library', 'n_study_environment_Quiet Room',
       'n_family_income_range_High', 'n_family_income_range_Low',
       'n_family_income_range_Medium', 'n_learning_style_Auditory',
       'n_learning_style_Kinesthetic', 'n_learning_style_Reading',
       'n_learning_style_Visual']]

In [67]:
y = df_f [['tutor']]

In [68]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=777)

In [69]:
X_train.shape,y_train.shape,X_test.shape,y_test.shape

((56000, 62), (56000, 1), (24000, 62), (24000, 1))

In [70]:
X_train.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['study_hours_per_day', 'social_media_hours', 'netflix_hours',
       'attendance_percentage', 'sleep_hours', 'mental_health_rating',
       'previous_gpa', 'stress_level', 'screen_time', 'time_management_score',
       'age', 'exercise_frequency', 'semester', 'social_activity',
       'parental_support_level', 'motivation_level', 'exam_anxiety_score',
       'exam_score', 'horas_de_ocio', 'actividad_total_diaria', 'manejo_de',
       'ratio_productividad_en_pantalla', 'estres_ansiedad',
       'manejo_de_estres', 'ratio_acceso_apoyo', 'aprovecamiento',
       'n_gender_Female', 'n_gender_Male', 'n_gender_Other', 'n_major_Arts',
       'n_major_Biology', 'n_major_Business', 'n_major_Computer Science',
       'n_major_Engineering', 'n_major_Psychology', 'n_part_time_job_No',
       'n_part_time_job_Yes', 'n_diet_quality_Fair', 'n_diet_quality_Good',
       'n_diet_quality_Poor', 'n_parental_education_level_Bachelor',
       'n_parental_educati

In [71]:
X_train.sample(10)

,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,mental_health_rating,previous_gpa,stress_level,screen_time,time_management_score,...,n_study_environment_Dorm,n_study_environment_Library,n_study_environment_Quiet Room,n_family_income_range_High,n_family_income_range_Low,n_family_income_range_Medium,n_learning_style_Auditory,n_learning_style_Kinesthetic,n_learning_style_Reading,n_learning_style_Visual
31296,7.10,2.10,1.60,64.10,7.40,6.50,4.00,1.40,11.20,3.20,...,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00
74997,0.90,4.50,2.50,80.40,5.00,6.00,2.69,7.60,9.40,9.70,...,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00
56309,2.70,3.00,2.00,100.00,4.80,7.60,3.82,3.80,9.00,3.40,...,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00
46390,4.20,0.30,3.30,61.00,7.90,9.20,4.00,6.00,8.00,2.30,...,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00
30473,5.10,0.40,0.80,73.30,8.60,5.40,3.08,6.00,7.30,9.10,...,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00
393,4.80,4.30,1.60,62.10,8.50,6.30,4.00,5.50,11.60,2.70,...,0.00,1.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00
343,2.00,4.50,0.40,41.90,6.60,6.70,2.60,10.00,7.10,1.60,...,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00
68192,2.28,1.60,0.60,44.10,7.70,7.70,3.16,6.70,6.10,1.70,...,0.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00
34815,9.40,3.20,4.00,97.80,7.20,7.20,3.20,4.20,16.70,6.50,...,1.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00
64363,3.62,0.80,2.40,69.40,10.00,9.40,3.60,5.80,7.90,3.40,...,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00


### StandardScaler

In [73]:
sc_xc = StandardScaler()
sc_yc = StandardScaler()
Xs_xc = pd.DataFrame(sc_xc.fit_transform(X_train),columns = X.columns)
Xs_yc = pd.DataFrame(sc_yc.fit_transform(y_train),columns = y.columns)

In [74]:
Xs_xc.columns

Index(['study_hours_per_day', 'social_media_hours', 'netflix_hours',
       'attendance_percentage', 'sleep_hours', 'mental_health_rating',
       'previous_gpa', 'stress_level', 'screen_time', 'time_management_score',
       'age', 'exercise_frequency', 'semester', 'social_activity',
       'parental_support_level', 'motivation_level', 'exam_anxiety_score',
       'exam_score', 'horas_de_ocio', 'actividad_total_diaria', 'manejo_de',
       'ratio_productividad_en_pantalla', 'estres_ansiedad',
       'manejo_de_estres', 'ratio_acceso_apoyo', 'aprovecamiento',
       'n_gender_Female', 'n_gender_Male', 'n_gender_Other', 'n_major_Arts',
       'n_major_Biology', 'n_major_Business', 'n_major_Computer Science',
       'n_major_Engineering', 'n_major_Psychology', 'n_part_time_job_No',
       'n_part_time_job_Yes', 'n_diet_quality_Fair', 'n_diet_quality_Good',
       'n_diet_quality_Poor', 'n_parental_education_level_Bachelor',
       'n_parental_education_level_High School',
       'n_paren

In [75]:
Xs_xc.head()

,study_hours_per_day,social_media_hours,netflix_hours,attendance_percentage,sleep_hours,mental_health_rating,previous_gpa,stress_level,screen_time,time_management_score,...,n_study_environment_Dorm,n_study_environment_Library,n_study_environment_Quiet Room,n_family_income_range_High,n_family_income_range_Low,n_family_income_range_Medium,n_learning_style_Auditory,n_learning_style_Kinesthetic,n_learning_style_Reading,n_learning_style_Visual
0,0.46,-0.76,1.56,0.78,-1.17,-0.94,0.86,-1.96,0.44,-0.85,...,-0.50,-0.50,2.00,1.41,-0.71,-0.71,-0.58,-0.58,-0.58,1.73
1,-2.08,0.90,0.87,1.69,1.29,0.73,0.86,-0.57,-0.74,-0.04,...,-0.50,-0.50,2.00,1.41,-0.71,-0.71,-0.58,-0.58,-0.58,1.73
2,1.43,1.04,0.95,-1.29,-1.10,0.83,0.86,-1.09,1.77,0.54,...,1.99,-0.50,-0.50,1.41,-0.71,-0.71,-0.58,-0.58,-0.58,1.73
3,1.21,1.59,0.09,0.82,-1.44,-0.42,-1.17,0.04,1.52,-0.88,...,-0.50,-0.50,-0.50,1.41,-0.71,-0.71,1.74,-0.58,-0.58,-0.58
4,0.91,0.28,0.18,-1.53,0.19,-0.68,-0.22,1.68,1.16,-0.04,...,-0.50,-0.50,2.00,-0.71,1.42,-0.71,-0.58,1.73,-0.58,-0.58


In [76]:
Xs_yc.head()

,tutor
0,-1.00
1,1.00
2,1.00
3,1.00
4,-1.00


## Modelado 

In [78]:
dc_scores ={}

### Regresión Lasso

In [80]:
model = Lasso()

In [81]:
model.get_params()

{'alpha': 1.0,
 'copy_X': True,
 'fit_intercept': True,
 'max_iter': 1000,
 'positive': False,
 'precompute': False,
 'random_state': None,
 'selection': 'cyclic',
 'tol': 0.0001,
 'warm_start': False}

In [82]:
model.fit(Xs_xc, Xs_yc)
ls_medias = cross_val_score(estimator=model, X=Xs_xc, y = Xs_yc, cv = 4, n_jobs=-1, scoring="r2")
np.mean(ls_medias), np.std(ls_medias)

(-0.00011223898983697689, 0.00012939794485927647)

In [83]:
#Combinación de parámetros
param_grid = {
    "alpha": [x for x in range(1, 100)] + [y/10 for y in range(10)],
    "tol": [0.00001, 0.0000001, 0.01],
    "selection": ['cyclic', 'random']
}

In [84]:
#Espacio de hiperparámetros
np.prod(list(map(len, param_grid.values())))

654

In [85]:
%%time
clf = GridSearchCV(model, param_grid, cv=4, error_score=-1000, n_jobs=-1, scoring="r2", verbose=5)
clf.fit(X_train, y_train)
print("Best score: " + str(clf.best_score_))

Fitting 4 folds for each of 654 candidates, totalling 2616 fits
[CV 2/4] END alpha=1, selection=cyclic, tol=1e-05;, score=0.002 total time=   0.0s
[CV 4/4] END alpha=1, selection=cyclic, tol=1e-05;, score=0.003 total time=   0.0s
[CV 1/4] END alpha=1, selection=cyclic, tol=1e-07;, score=0.002 total time=   0.0s
[CV 3/4] END alpha=1, selection=cyclic, tol=1e-07;, score=0.002 total time=   0.0s
[CV 4/4] END alpha=1, selection=cyclic, tol=1e-07;, score=0.003 total time=   0.0s
[CV 1/4] END alpha=1, selection=cyclic, tol=0.01;, score=0.002 total time=   0.0s
[CV 2/4] END alpha=1, selection=cyclic, tol=0.01;, score=0.002 total time=   0.0s
[CV 3/4] END alpha=1, selection=cyclic, tol=0.01;, score=0.002 total time=   0.0s
[CV 4/4] END alpha=1, selection=cyclic, tol=0.01;, score=0.003 total time=   0.0s
[CV 3/4] END alpha=1, selection=random, tol=1e-05;, score=0.002 total time=   0.0s
[CV 2/4] END alpha=1, selection=random, tol=1e-05;, score=0.002 total time=   0.1s
[CV 4/4] END alpha=1, selec

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py:1151: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py:1151: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
/Users/whiz/anaconda

[CV 3/4] END alpha=99, selection=random, tol=0.01;, score=-0.000 total time=   0.0s
[CV 2/4] END alpha=99, selection=random, tol=0.01;, score=-0.000 total time=   0.0s
[CV 4/4] END alpha=99, selection=random, tol=0.01;, score=-0.000 total time=   0.0s


/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py:1151: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py:1151: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
/Users/whiz/anaconda

[CV 1/4] END alpha=0.0, selection=cyclic, tol=1e-05;, score=0.785 total time=   3.5s


...

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.125e+03, tolerance: 1.050e-03 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


[CV 2/4] END alpha=0.8, selection=cyclic, tol=1e-05;, score=0.003 total time=   0.0s
[CV 3/4] END alpha=0.8, selection=cyclic, tol=1e-05;, score=0.003 total time=   0.0s
[CV 4/4] END alpha=0.8, selection=cyclic, tol=1e-05;, score=0.004 total time=   0.0s
[CV 1/4] END alpha=0.8, selection=cyclic, tol=1e-07;, score=0.002 total time=   0.0s
[CV 2/4] END alpha=0.8, selection=cyclic, tol=1e-07;, score=0.003 total time=   0.0s
[CV 3/4] END alpha=0.8, selection=cyclic, tol=1e-07;, score=0.003 total time=   0.0s
[CV 4/4] END alpha=0.8, selection=cyclic, tol=1e-07;, score=0.004 total time=   0.0s
[CV 1/4] END alpha=0.8, selection=cyclic, tol=0.01;, score=0.002 total time=   0.0s
[CV 2/4] END alpha=0.8, selection=cyclic, tol=0.01;, score=0.003 total time=   0.0s
[CV 3/4] END alpha=0.8, selection=cyclic, tol=0.01;, score=0.003 total time=   0.0s
[CV 4/4] END alpha=0.8, selection=cyclic, tol=0.01;, score=0.004 total time=   0.0s
[CV 1/4] END alpha=0.8, selection=random, tol=1e-05;, score=0.002 tot

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.127e+03, tolerance: 1.050e+02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.125e+03, tolerance: 1.050e+02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented 

[CV 4/4] END alpha=0.0, selection=random, tol=0.01;, score=0.784 total time=   3.8s
Best score: 0.7850849392694896
CPU times: user 1min 28s, sys: 18.6 s, total: 1min 46s
Wall time: 58.9 s


/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.502e+03, tolerance: 1.400e-03 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


In [86]:
summary = pd.DataFrame(clf.cv_results_)

In [87]:
summary.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_alpha,param_selection,param_tol,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,mean_test_score,std_test_score,rank_test_score
0,0.04,0.01,0.00,0.00,1,cyclic,0.00,"{'alpha': 1, 'selection': 'cyclic', 'tol': 1e-05}",0.00,0.00,0.00,0.00,0.00,0.00,61
1,0.03,0.01,0.00,0.00,1,cyclic,0.00,"{'alpha': 1, 'selection': 'cyclic', 'tol': 1e-07}",0.00,0.00,0.00,0.00,0.00,0.00,61
2,0.03,0.01,0.00,0.00,1,cyclic,0.01,"{'alpha': 1, 'selection': 'cyclic', 'tol': 0.01}",0.00,0.00,0.00,0.00,0.00,0.00,61
3,0.04,0.01,0.01,0.00,1,random,0.00,"{'alpha': 1, 'selection': 'random', 'tol': 1e-05}",0.00,0.00,0.00,0.00,0.00,0.00,61
4,0.04,0.01,0.01,0.00,1,random,0.00,"{'alpha': 1, 'selection': 'random', 'tol': 1e-07}",0.00,0.00,0.00,0.00,0.00,0.00,61


In [88]:
summary.sort_values(by = "rank_test_score")

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_alpha,param_selection,param_tol,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,mean_test_score,std_test_score,rank_test_score
598,3.51,0.24,0.01,0.00,0.00,random,0.00,"{'alpha': 0.0, 'selection': 'random', 'tol': 1...",0.78,0.79,0.79,0.78,0.79,0.00,1
597,4.36,0.22,0.01,0.00,0.00,random,0.00,"{'alpha': 0.0, 'selection': 'random', 'tol': 1...",0.78,0.79,0.79,0.78,0.79,0.00,2
599,4.12,0.30,0.00,0.00,0.00,random,0.01,"{'alpha': 0.0, 'selection': 'random', 'tol': 0...",0.78,0.79,0.79,0.78,0.79,0.00,3
595,4.09,0.05,0.01,0.00,0.00,cyclic,0.00,"{'alpha': 0.0, 'selection': 'cyclic', 'tol': 1...",0.78,0.79,0.79,0.78,0.79,0.00,4
596,4.92,0.30,0.00,0.00,0.00,cyclic,0.01,"{'alpha': 0.0, 'selection': 'cyclic', 'tol': 0...",0.78,0.79,0.79,0.78,0.79,0.00,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,0.03,0.00,0.00,0.00,44,cyclic,0.00,"{'alpha': 44, 'selection': 'cyclic', 'tol': 1e...",-0.00,-0.00,-0.00,-0.00,-0.00,0.00,73
260,0.03,0.00,0.00,0.00,44,cyclic,0.01,"{'alpha': 44, 'selection': 'cyclic', 'tol': 0.01}",-0.00,-0.00,-0.00,-0.00,-0.00,0.00,73
261,0.02,0.00,0.00,0.00,44,random,0.00,"{'alpha': 44, 'selection': 'random', 'tol': 1e...",-0.00,-0.00,-0.00,-0.00,-0.00,0.00,73
263,0.03,0.01,0.00,0.00,44,random,0.01,"{'alpha': 44, 'selection': 'random', 'tol': 0.01}",-0.00,-0.00,-0.00,-0.00,-0.00,0.00,73


In [89]:
dc_scores[str(model).split("(")[0]] = {"model": clf.best_estimator_, "score": clf.best_score_}

In [90]:
dc_scores

{'Lasso': {'model': Lasso(alpha=0.0, selection='random', tol=1e-07),
  'score': 0.7850849392694896}}

### Ridge Regression

In [92]:
model = Ridge()

In [93]:
model.fit(X_train, y_train)
ls_medias = cross_val_score(estimator=model, X=X_test, y = y_test, cv = 4, n_jobs=-1, scoring="r2")
np.mean(ls_medias), np.std(ls_medias)

(0.7843214475621396, 0.0011700487895043939)

In [94]:
model.get_params()

{'alpha': 1.0,
 'copy_X': True,
 'fit_intercept': True,
 'max_iter': None,
 'positive': False,
 'random_state': None,
 'solver': 'auto',
 'tol': 0.0001}

In [95]:
param_grid = {
    "alpha": [x for x in range(1, 100)] + [y/10 for y in range(10)],
    "tol": [0.0000001, 0.01],
    "solver": ['auto', 'svd', 'cholesky', 'lsqr'],
    "max_iter": [500]
}

In [96]:
np.prod(list(map(len, param_grid.values())))

872

In [97]:
%%time
clf = GridSearchCV(model, param_grid, cv=4, error_score=-1000, n_jobs=-1, scoring="r2")
clf.fit(X_train, y_train)
print("Best score: " + str(clf.best_score_))

/Users/whiz/anaconda3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best score: 0.7850892844834929
CPU times: user 4.35 s, sys: 9.36 s, total: 13.7 s
Wall time: 1min 11s


In [98]:
summary = pd.DataFrame(clf.cv_results_)

In [99]:
summary.sort_values(by = "rank_test_score")

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_alpha,param_max_iter,param_solver,param_tol,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,mean_test_score,std_test_score,rank_test_score
620,0.04,0.01,0.01,0.00,78,500,cholesky,0.00,"{'alpha': 78, 'max_iter': 500, 'solver': 'chol...",0.78,0.79,0.79,0.78,0.79,0.00,1
618,0.19,0.02,0.01,0.00,78,500,svd,0.00,"{'alpha': 78, 'max_iter': 500, 'solver': 'svd'...",0.78,0.79,0.79,0.78,0.79,0.00,1
617,0.03,0.00,0.00,0.00,78,500,auto,0.01,"{'alpha': 78, 'max_iter': 500, 'solver': 'auto...",0.78,0.79,0.79,0.78,0.79,0.00,1
616,0.03,0.00,0.00,0.00,78,500,auto,0.00,"{'alpha': 78, 'max_iter': 500, 'solver': 'auto...",0.78,0.79,0.79,0.78,0.79,0.00,1
621,0.03,0.01,0.01,0.00,78,500,cholesky,0.01,"{'alpha': 78, 'max_iter': 500, 'solver': 'chol...",0.78,0.79,0.79,0.78,0.79,0.00,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
759,0.04,0.01,0.01,0.00,95,500,lsqr,0.01,"{'alpha': 95, 'max_iter': 500, 'solver': 'lsqr...",0.78,0.78,0.78,0.78,0.78,0.00,868
767,0.04,0.00,0.01,0.00,96,500,lsqr,0.01,"{'alpha': 96, 'max_iter': 500, 'solver': 'lsqr...",0.78,0.78,0.78,0.78,0.78,0.00,869
775,0.04,0.00,0.01,0.00,97,500,lsqr,0.01,"{'alpha': 97, 'max_iter': 500, 'solver': 'lsqr...",0.78,0.78,0.78,0.78,0.78,0.00,870
783,0.04,0.00,0.01,0.00,98,500,lsqr,0.01,"{'alpha': 98, 'max_iter': 500, 'solver': 'lsqr...",0.78,0.78,0.78,0.78,0.78,0.00,871


In [100]:
dc_scores[str(model).split("(")[0]] = {"model": clf.best_estimator_, "score": clf.best_score_}

In [101]:
dc_scores

{'Lasso': {'model': Lasso(alpha=0.0, selection='random', tol=1e-07),
  'score': 0.7850849392694896},
 'Ridge': {'model': Ridge(alpha=78, max_iter=500, tol=1e-07),
  'score': 0.7850892844834929}}

### Elastic Net

In [103]:
model = ElasticNet()

In [104]:
model.fit(X_train, y_train)
ls_medias = cross_val_score(estimator=model, X=X_test, y = y_test, cv = 4, n_jobs=-1, scoring="r2")
np.mean(ls_medias), np.std(ls_medias)

(0.12022711778861864, 0.001776390571775688)

In [105]:
model.get_params()

{'alpha': 1.0,
 'copy_X': True,
 'fit_intercept': True,
 'l1_ratio': 0.5,
 'max_iter': 1000,
 'positive': False,
 'precompute': False,
 'random_state': None,
 'selection': 'cyclic',
 'tol': 0.0001,
 'warm_start': False}

In [106]:
param_grid = {
    "alpha": [x for x in range(1, 10)] + [y/10 for y in range(10)],
    "l1_ratio": [x for x in range(1, 10)] + [y/10 for y in range(10)],
    "selection": ["cyclic", "random"]
}

In [107]:
np.prod(list(map(len, param_grid.values())))

722

In [108]:
%%time
clf = GridSearchCV(model, param_grid, cv=4, error_score=-1000, n_jobs=-1, scoring="r2", verbose=5)
clf.fit(X_train, y_train)
print("Best score: " + str(clf.best_score_))

Fitting 4 folds for each of 722 candidates, totalling 2888 fits
[CV 2/4] END alpha=1, l1_ratio=1, selection=cyclic;, score=0.002 total time=   0.0s
[CV 1/4] END alpha=1, l1_ratio=1, selection=cyclic;, score=0.002 total time=   0.0s
[CV 3/4] END alpha=1, l1_ratio=1, selection=cyclic;, score=0.002 total time=   0.0s
[CV 4/4] END alpha=1, l1_ratio=1, selection=cyclic;, score=0.003 total time=   0.0s
[CV 1/4] END alpha=1, l1_ratio=1, selection=random;, score=0.002 total time=   0.0s
[CV 2/4] END alpha=1, l1_ratio=1, selection=random;, score=0.002 total time=   0.1s
[CV 3/4] END alpha=1, l1_ratio=1, selection=random;, score=0.002 total time=   0.0s
[CV 4/4] END alpha=1, l1_ratio=1, selection=random;, score=0.003 total time=   0.0s
[CV 1/4] END alpha=1, l1_ratio=2, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 2/4] END alpha=1, l1_ratio=2, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 3/4] END alpha=1, l1_ratio=2, selection=cyclic;, score=-1000.000 total time=   0.0s


/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.689e+03, tolerance: 1.050e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


[CV 3/4] END alpha=2, l1_ratio=9, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 4/4] END alpha=2, l1_ratio=9, selection=cyclic;, score=-1000.000 total time=   0.0s
[CV 3/4] END alpha=1, l1_ratio=0.0, selection=cyclic;, score=0.662 total time=   4.1s
[CV 1/4] END alpha=2, l1_ratio=9, selection=random;, score=-1000.000 total time=   0.0s
[CV 2/4] END alpha=2, l1_ratio=9, selection=random;, score=-1000.000 total time=   0.0s
[CV 3/4] END alpha=2, l1_ratio=9, selection=random;, score=-1000.000 total time=   0.0s
[CV 4/4] END alpha=2, l1_ratio=9, selection=random;, score=-1000.000 total time=   0.0s


/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.689e+03, tolerance: 1.050e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.691e+03, tolerance: 1.050e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented 

[CV 1/4] END alpha=1, l1_ratio=0.0, selection=random;, score=0.663 total time=   4.4s
[CV 1/4] END alpha=1, l1_ratio=0.0, selection=cyclic;, score=0.663 total time=   4.5s


...

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.593e+03, tolerance: 1.050e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


[CV 2/4] END alpha=0.9, l1_ratio=0.4, selection=cyclic;, score=0.196 total time=   0.0s
[CV 3/4] END alpha=0.9, l1_ratio=0.4, selection=cyclic;, score=0.196 total time=   0.0s
[CV 4/4] END alpha=0.9, l1_ratio=0.4, selection=cyclic;, score=0.196 total time=   0.0s
[CV 1/4] END alpha=0.9, l1_ratio=0.4, selection=random;, score=0.197 total time=   0.0s
[CV 2/4] END alpha=0.9, l1_ratio=0.4, selection=random;, score=0.196 total time=   0.0s
[CV 3/4] END alpha=0.9, l1_ratio=0.4, selection=random;, score=0.196 total time=   0.0s
[CV 4/4] END alpha=0.9, l1_ratio=0.4, selection=random;, score=0.196 total time=   0.1s
[CV 1/4] END alpha=0.9, l1_ratio=0.5, selection=cyclic;, score=0.156 total time=   0.0s
[CV 2/4] END alpha=0.9, l1_ratio=0.5, selection=cyclic;, score=0.156 total time=   0.0s
[CV 3/4] END alpha=0.9, l1_ratio=0.5, selection=cyclic;, score=0.156 total time=   0.0s
[CV 4/4] END alpha=0.9, l1_ratio=0.5, selection=cyclic;, score=0.157 total time=   0.0s
[CV 1/4] END alpha=0.9, l1_ratio

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:425: FitFailedWarning: 
1216 fits failed out of a total of 2888.
The score on these train-test partitions for these parameters will be set to -1000.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
152 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 732, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py", line 1144, in wrapper
    estimator._validate_params()
  File "/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/base.py", line 637, in _validate_params
    validate_parameter_constraints(
  File "/Use

Best score: 0.7850849481216164
CPU times: user 1min 35s, sys: 38.5 s, total: 2min 13s
Wall time: 3min 42s


/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.502e+03, tolerance: 1.400e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


In [109]:
summary = pd.DataFrame(clf.cv_results_)

In [110]:
dc_scores[str(model).split("(")[0]] = {"model": clf.best_estimator_, "score": clf.best_score_}

In [111]:
dc_scores

{'Lasso': {'model': Lasso(alpha=0.0, selection='random', tol=1e-07),
  'score': 0.7850849392694896},
 'Ridge': {'model': Ridge(alpha=78, max_iter=500, tol=1e-07),
  'score': 0.7850892844834929},
 'ElasticNet': {'model': ElasticNet(alpha=0.0, l1_ratio=0.6, selection='random'),
  'score': 0.7850849481216164}}

In [112]:
print("Best score: " + str(clf.best_score_))

Best score: 0.7850849481216164


In [113]:
#mejor modelo
clf.best_estimator_

ElasticNet(alpha=0.0, l1_ratio=0.6, selection='random')

In [114]:
#mejores parámetros
clf.best_params_

{'alpha': 0.0, 'l1_ratio': 0.6, 'selection': 'random'}

### Support Vector Machine

In [116]:
svm = SVC()
svm.fit(X_train, y_train.values.ravel())
ls_scores = cross_val_score(estimator=svm, scoring="accuracy", X=X_test, y = y_test, cv=4, n_jobs=-1)
np.mean(ls_scores), np.std(ls_scores)

/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1184: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1184: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1184: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/whiz/anaconda3/lib/python3.11/site-packages/sklearn/utils/validation.py:1184: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please chan

(0.803375, 0.002286357486386511)

In [271]:
svm.get_params()

{'C': 1.0,
 'break_ties': False,
 'cache_size': 200,
 'class_weight': None,
 'coef0': 0.0,
 'decision_function_shape': 'ovr',
 'degree': 3,
 'gamma': 'scale',
 'kernel': 'rbf',
 'max_iter': -1,
 'probability': False,
 'random_state': None,
 'shrinking': True,
 'tol': 0.001,
 'verbose': False}